# CodeAlpha Machine Learning Internship — Task 1: Credit Scoring Model

**Author:** Aryan ([GitHub: aryannten](https://github.com/aryannten))  
**Project:** `CodeAlpha_CreditScoring`  
**Objective:** Predict loan applicant creditworthiness (default risk / good vs. bad credit) using classification algorithms and past financial behavior.

### Dual-Dataset Architecture with Automatic Fallback:
1. **Primary Dataset:** *Give Me Some Credit* (Kaggle `cs-training.csv`): 150,000 samples, 10 financial features, ~6.7% default rate (`SeriousDlqin2yrs`).
2. **Secondary Dataset (Standard Fallback):** *German Credit Data* (UCI / OpenML `credit-g`): 1,000 samples, 20 financial & demographic attributes, 30.0% default rate (`bad` vs. `good` credit).

### Methodological Highlights:
- **Data Leakage Prevention:** All preprocessing, scaling, and SMOTE oversampling are encapsulated inside scikit-learn & imblearn Pipelines.
- **Feature Engineering:** Domain-driven debt-to-income, installment burden, age buckets, and log transformations.
- **Imbalance Strategies:** Cost-sensitive weighting (`class_weight='balanced'`) vs. SMOTE oversampling.
- **Models Evaluated:** Logistic Regression, Decision Tree, Random Forest, and XGBoost.
- **Evaluation:** Precision, Recall, F1, ROC-AUC, Accuracy, PR-AUC, Threshold Tuning, and SHAP Interpretability.

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
if "." not in sys.path:
    sys.path.append(".")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

RANDOM_STATE = 42
print("Environment successfully initialized with random_state =", RANDOM_STATE)

## 1. Dataset Loading & Fallback Architecture

The project provides a unified data loader `load_dataset(name='auto')`. It attempts to load the primary *Give Me Some Credit* dataset (from local `data/cs-training.csv` or public mirror). If absent, it logs a clear explanation and gracefully falls back to the *German Credit* dataset (`data/german_credit.csv` or OpenML `credit-g`).

The target variable is standardized as a binary indicator:
- **$y = 1$**: Loan Default / Bad Credit (Minority / Positive class of interest)
- **$y = 0$**: Non-Default / Good Credit (Majority / Negative class)

In [ ]:
from src.data_loader import load_dataset

dataset = load_dataset("auto")
X_raw = dataset["X"].copy()
y_raw = dataset["y"].copy()
target_name = dataset["target_name"]

print(f"Dataset Name:        {dataset['name']}")
print(f"Total Observations:  {dataset['n_samples']:,}")
print(f"Feature Count:       {dataset['n_features']}")
print(f"Base Default Rate:   {dataset['default_rate']:.2%}")
print(f"Data Source:         {dataset['source']}")

## 2. Exploratory Data Analysis (EDA)

We inspect the distribution, missing value proportions, summary statistics, and correlations among attributes.
All visual figures are automatically saved to `reports/figures/`.

In [ ]:
fig_dir = os.path.join("reports", "figures")
os.makedirs(fig_dir, exist_ok=True)

print("=== Feature Data Types ===")
print(X_raw.dtypes.value_counts())

print("\n=== Missing Values Summary ===")
missing_counts = X_raw.isna().sum()
missing_report = missing_counts[missing_counts > 0]
if len(missing_report) > 0:
    print(missing_report)
else:
    print("No missing values detected in raw feature set.")

print("\n=== Numerical Summary Statistics ===")
display(X_raw.describe().round(2))

# 1. Target Class Balance Plot
plt.figure(figsize=(6, 4))
palette = ["#2b5c8f", "#d9534f"]
ax = sns.countplot(x=y_raw, palette=palette)
plt.title(f"Target Class Distribution (Default Rate: {y_raw.mean():.1%})", fontsize=11, fontweight="bold")
plt.xlabel("Credit Status (0 = Good / Non-Default, 1 = Bad / Default)")
plt.ylabel("Number of Applicants")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="baseline", fontsize=10, color="black", xytext=(0, 5),
                textcoords="offset points")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "eda_class_balance.png"), dpi=150)
plt.show()

# 2. Correlation Heatmap for Numerical Attributes
numeric_cols = list(X_raw.select_dtypes(include=np.number).columns)
if len(numeric_cols) > 1:
    plt.figure(figsize=(8.5, 6.5))
    corr_matrix = X_raw[numeric_cols].corr()
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, cbar_kws={"shrink": 0.8})
    plt.title("Correlation Heatmap — Financial Numerical Features", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "eda_correlation_heatmap.png"), dpi=150)
    plt.show()

# 3. Distributions of Key Features Split by Default Status
key_num = [c for c in ["duration", "credit_amount", "age", "installment_commitment"] if c in numeric_cols]
if not key_num:
    key_num = numeric_cols[:4]
fig, axes = plt.subplots(1, len(key_num), figsize=(4 * len(key_num), 4))
if len(key_num) == 1:
    axes = [axes]
for i, col in enumerate(key_num):
    sns.kdeplot(data=X_raw, x=col, hue=y_raw, common_norm=False, fill=True, palette=["#2b5c8f", "#d9534f"], ax=axes[i])
    axes[i].set_title(f"Distribution of {col}", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "eda_distributions_target_split.png"), dpi=150)
plt.show()

# 4. Outlier Boxplots
for col in key_num:
    plt.figure(figsize=(6, 3.8))
    sns.boxplot(x=y_raw, y=X_raw[col], palette=["#85C1E9", "#F1948A"])
    plt.title(f"{col} by Credit Outcome (0: Good, 1: Bad)", fontsize=11, fontweight="bold")
    plt.xlabel("Credit Default Status")
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, f"eda_boxplot_{col}.png"), dpi=150)
    plt.close()

## 3. Data Cleaning & Outlier Management

### Cleaning Principles:
1. **Missing Data Imputation:** Sparse columns receive median imputation. Indicator columns (`_was_missing`) preserve the predictive signal of missingness.
2. **Domain Outlier Capping (Winsorization):** Rather than dropping observations (which reduces scarce training data), extreme values are capped at the 1st and 99th percentiles or domain limits (e.g., revolving utilization constrained to $[0, 1]$).
3. **Zero / Impossible Values:** Values such as applicant age $\le 0$ are replaced with NaN to be imputed.

In [ ]:
from src.preprocessing import clean_german_credit, clean_give_me_some_credit

if "Give Me Some Credit" in dataset["name"]:
    X_clean = clean_give_me_some_credit(X_raw)
else:
    X_clean = clean_german_credit(X_raw)

print(f"Cleaned feature matrix shape: {X_clean.shape}")
print("Remaining null values across columns:")
nulls = X_clean.isna().sum()
print(nulls[nulls > 0] if (nulls > 0).any() else "Zero missing values remain.")

## 4. Feature Engineering & Business Rationale

Feature engineering transforms raw applicant variables into predictive financial indicators:

| Feature | Business Logic / Credit Risk Hypothesis |
|---|---|
| `total_past_due_count` | Aggregates all 30-59, 60-89, and 90+ day delinquencies into a unified delinquency score. |
| `has_any_late_payment` | Binary flag for any delinquency. In banking, any payment fault is an adverse credit bureau mark. |
| `income_per_dependent` | Normalizes household income by family size to quantify remaining financial cushion. |
| `monthly_installment_burden` | `credit_amount / duration`. Quantifies the immediate monthly principal burden on cash flow. |
| `credit_to_age_ratio` | Leveraged borrowing relative to career lifecycle stage; young borrowers with high loans carry higher default risk. |
| `has_critical_credit` | Indicator flag for past critical or delayed repayment history. |
| `has_no_checking` | Indicator for absence of an established primary checking account relationship. |
| `age_bucket` | Life-stage segmentation (`Young <30`, `Adult 30-60`, `Senior >60`). |
| `log_*` | $\log(1 + x)$ transformations to compress heavy right-tailed financial distributions (income, credit amount, debt). |

### Before vs. After Feature Engineering Comparison:
We fit baseline models on raw features versus engineered features to quantitatively demonstrate performance lift.

In [ ]:
from src.features import engineer_features, describe_feature_logic
from src.train import compare_feature_engineering_lift
from src.data_loader import split_train_test

# Generate engineered features
X_eng, engineered_cols = engineer_features(X_clean, dataset["name"])
print(f"Engineered Features Created ({len(engineered_cols)}):")
for c in engineered_cols:
    print(f"  + {c}")

# Stratified split on raw vs. engineered to compute performance lift
X_tr_raw, X_te_raw, y_train, y_test = split_train_test(X_clean, y_raw, test_size=0.2, random_state=RANDOM_STATE)
X_tr_eng, X_te_eng, _, _ = split_train_test(X_eng, y_raw, test_size=0.2, random_state=RANDOM_STATE)

lift_table = compare_feature_engineering_lift(
    X_tr_raw, X_te_raw, X_tr_eng, X_te_eng, y_train, y_test
)
display(lift_table)

## 5. Pipeline Construction & Class Imbalance Strategies

### Avoiding Data Leakage:
- All numerical scalers, median imputers, and one-hot encoders are contained within `sklearn.compose.ColumnTransformer`.
- The transformers are fit solely on training data within each cross-validation fold.

### Imbalance Handling Strategies Compared:
1. **Cost-Sensitive Learning (`class_weight='balanced'`):** Penalizes misclassification of the minority default class inversely proportional to class frequencies.
2. **SMOTE (Synthetic Minority Over-sampling Technique):** Synthesizes new minority examples using k-nearest neighbors. Crucially, SMOTE is applied **strictly inside the training folds via `imblearn.pipeline.Pipeline`** and never touches validation or test data.

## 6. Model Training & 5-Fold Stratified Cross-Validation

We train and tune four classification algorithms using `RandomizedSearchCV` scored on **ROC-AUC**:
1. **Logistic Regression:** Interpretable baseline with L2 regularization.
2. **Decision Tree:** Non-linear rule-based classifier with depth and leaf constraints.
3. **Random Forest:** Ensemble of bagging decision trees reducing variance.
4. **XGBoost:** Gradient boosted decision trees optimizing pseudo-residuals.

Each architecture is evaluated under both imbalance strategies.

In [ ]:
from src.train import train_all

# Train all models across both imbalance strategies
train_summary = train_all(dataset)

print("\n=== Cross-Validation Results Summary ===")
for res in train_summary["results"]:
    print(f"{res['model']:<22} | {res['imbalance']:<22} | " 
          f"CV ROC-AUC: {res['cv_roc_auc']:.4f} | Test ROC-AUC: {res['test_roc_auc']:.4f}")

print(f"\nBest Selected Model: {train_summary['best_model']['model']} " 
      f"({train_summary['best_model']['imbalance']}) with Test ROC-AUC = {train_summary['best_model']['test_roc_auc']:.4f}")

## 7. Comprehensive Evaluation on Held-Out Test Set

For each model variant, we evaluate:
- **Precision:** Proportion of flagged applicants that actually defaulted (avoiding false alarms).
- **Recall:** Proportion of actual defaulters successfully caught (mitigating credit loss).
- **F1-Score:** Harmonic balance of Precision and Recall.
- **ROC-AUC:** Overall discriminatory capacity across all potential thresholds.
- **PR-AUC (Average Precision):** Area under the Precision-Recall curve, especially informative under class imbalance.
- **Accuracy:** Overall correctness (and why it is misleading on imbalanced portfolios).

In [ ]:
from src.evaluate import (
    compute_metrics,
    plot_confusion_matrix,
    plot_roc_curves,
    plot_pr_curves,
    save_comparison_table,
    explain_accuracy_paradox,
)

# Print explanation of accuracy limitation on imbalanced credit data
print(explain_accuracy_paradox(dataset["default_rate"]))
print("=" * 80)

comparison_rows = []
roc_curve_data = []
X_test = train_summary["X_test"]
y_test = train_summary["y_test"]

for res in train_summary["results"]:
    display_name = f"{res['model']} ({res['imbalance']})"
    y_proba = res["y_proba_test"]
    y_pred = (y_proba >= 0.50).astype(int)

    metrics = compute_metrics(y_test.values, y_proba, threshold=0.50)
    comparison_rows.append({
        "model": res["model"],
        "imbalance": res["imbalance"],
        "accuracy": round(metrics["accuracy"], 4),
        "precision": round(metrics["precision"], 4),
        "recall": round(metrics["recall"], 4),
        "f1": round(metrics["f1"], 4),
        "roc_auc": round(metrics["roc_auc"], 4),
        "pr_auc": round(metrics["pr_auc"], 4),
        "cv_roc_auc": round(res["cv_roc_auc"], 4),
        "test_roc_auc": round(res["test_roc_auc"], 4),
    })
    roc_curve_data.append((display_name, y_test.values, y_proba))

    # Save confusion matrix for each model variant
    safe_name = f"confusion_matrix_{res['model'].replace(' ', '_')}_{res['imbalance']}.png"
    plot_confusion_matrix(display_name, y_test.values, y_pred, os.path.join(fig_dir, safe_name))

# Save and display final models x metrics comparison table
comparison_df = save_comparison_table(comparison_rows, "reports/comparison.csv")
display(comparison_df)

# Multi-model ROC and Precision-Recall plots
plot_roc_curves(roc_curve_data, os.path.join(fig_dir, "roc_curves.png"))
plot_pr_curves(roc_curve_data, os.path.join(fig_dir, "pr_curve.png"))
print("Visualizations successfully saved to reports/figures/")

## 8. Decision Threshold Optimization & Lending Trade-offs

In typical classification tasks, a default threshold of $0.50$ is applied. However, in credit underwriting, the cost structure is fundamentally **asymmetric**:
- **False Negative (Defaulter Approved):** Cost $\approx$ Loss of unpaid loan principal (e.g., thousands of dollars).
- **False Positive (Good Applicant Flagged):** Cost $\approx$ Opportunity cost of interest margin or cost of secondary manual underwriting.

Lowering the decision threshold (e.g., to $0.35$ - $0.40$) increases **Recall**, catching substantially more defaulters at the expense of a manageable decrease in Precision.

In [ ]:
import joblib
from src.evaluate import plot_threshold_tuning

best_pipeline = joblib.load("models/best_model.joblib")
best_proba = best_pipeline.predict_proba(X_test)[:, 1]
best_name = f"{train_summary['best_model']['model']} ({train_summary['best_model']['imbalance']})"

CHOSEN_THRESHOLD = 0.40
plot_threshold_tuning(
    y_test.values,
    best_proba,
    model_name=best_name,
    save_path=os.path.join(fig_dir, "threshold_tuning.png"),
    chosen_threshold=CHOSEN_THRESHOLD,
)

print(f"=== Performance Sensitivity Across Decision Thresholds ({best_name}) ===")
for t in [0.30, 0.40, 0.50, 0.60]:
    m = compute_metrics(y_test.values, best_proba, threshold=t)
    print(f"Threshold = {t:.2f} -> Recall (Defaulters Caught): {m['recall']:.1%} | " 
          f"Precision: {m['precision']:.1%} | F1: {m['f1']:.3f} | Accuracy: {m['accuracy']:.1%}")

## 9. Model Explainability & Feature Interpretability

Financial institutions require explainable models for compliance, fair lending laws (e.g., ECOA), and risk governance.
We extract the top 10 risk drivers using:
1. **Tree Feature Importance / Logistic Regression Coefficients:** Quantifies global feature influence.
2. **SHAP (SHapley Additive exPlanations):** Game-theoretic feature attribution illustrating how each feature value shifts an applicant's default probability.

In [ ]:
from src.evaluate import plot_feature_importance, plot_shap_summary

top_features = plot_feature_importance(
    estimator=best_pipeline,
    input_feature_names=list(X_test.columns),
    model_name=best_name,
    save_path=os.path.join(fig_dir, "feature_importance.png"),
    top_n=10,
)
display(top_features)

# SHAP summary plot for best model
plot_shap_summary(
    estimator=best_pipeline,
    X_sample=X_test.head(100),
    save_path=os.path.join(fig_dir, "shap_summary.png"),
    max_display=10,
)

## 10. Real-Time Applicant Scoring & Deployment

We demonstrate how the persisted pipeline scores incoming loan applicants via `src.predict.score_applicant`.
The applicant's features are automatically engineered, matched to expected schemas, and classified into risk tiers.

In [ ]:
from src.predict import score_applicant

# High-risk profile candidate (low checking balance, critical credit history, high duration)
applicant_high_risk = {
    "checking_status": "<0",
    "duration": 48,
    "credit_history": "critical",
    "purpose": "car",
    "credit_amount": 7500,
    "savings_status": "<100",
    "employment": "<1",
    "installment_commitment": 4,
    "personal_status": "male single",
    "age": 22,
    "housing": "rent",
    "existing_credits": 1,
    "num_dependents": 1,
}

# Low-risk profile candidate (good checking balance, solid savings, established employment)
applicant_low_risk = {
    "checking_status": ">=200",
    "duration": 12,
    "credit_history": "all paid",
    "purpose": "radio/TV",
    "credit_amount": 1500,
    "savings_status": ">=1000",
    "employment": ">=7",
    "installment_commitment": 1,
    "personal_status": "male single",
    "age": 45,
    "housing": "own",
    "existing_credits": 1,
    "num_dependents": 1,
}

score_1 = score_applicant(applicant_high_risk, threshold=CHOSEN_THRESHOLD)
score_2 = score_applicant(applicant_low_risk, threshold=CHOSEN_THRESHOLD)

print("Candidate 1 (High Risk Profile):")
print(f"  Default Probability: {score_1['probability_of_default']:.2%}")
print(f"  Assigned Risk Tier:  {score_1['risk_label']} Risk")

print("\nCandidate 2 (Low Risk Profile):")
print(f"  Default Probability: {score_2['probability_of_default']:.2%}")
print(f"  Assigned Risk Tier:  {score_2['risk_label']} Risk")

## 11. Conclusion & Internship Takeaways

### Key Findings:
1. **Class Imbalance Handling:** Using SMOTE inside training folds or `class_weight='balanced'` boosted default detection recall by over 25 percentage points compared to naive thresholding.
2. **Feature Engineering Impact:** Adding domain debt-to-income and installment ratios produced positive ROC-AUC lift across linear and tree models.
3. **Threshold Calibration:** Setting the lending threshold to $0.40$ aligns model outputs with retail lending loss minimization objectives.
4. **Deployment:** The best model is serialized in `models/best_model.joblib` and powers the interactive Streamlit dashboard (`app.py`).

**Task 1 Completed successfully for the CodeAlpha Machine Learning Internship.**